In [2]:
import h5py


def print_hdf5_tree(file_path_or_obj, show_attrs=False):
    """HDF5 파일의 그룹 및 데이터셋 구조를 트리 형태로 출력합니다.

    :param file_path_or_obj: HDF5 파일 경로(str) 또는 이미 열린 h5py.File/Group
    객체 :param show_attrs: 속성(Attributes) 정보 포함 여부 (기본값: False)
    """

    def _visit_node(name, obj, prefix="", is_last=True):
        connector = "└── " if is_last else "├── "
        display_name = name.split("/")[-1]

        if isinstance(obj, h5py.Dataset):
            info = f"{display_name} [Dataset: shape={obj.shape}, dtype={obj.dtype}]"
            print(f"{prefix}{connector}{info}")
        elif isinstance(obj, h5py.Group):
            print(f"{prefix}{connector}{display_name}/")

        # 속성(Attribute) 출력 옵션
        if show_attrs and len(obj.attrs) > 0:
            attr_prefix = prefix + ("    " if is_last else "│   ")
            attr_keys = list(obj.attrs.keys())
            for idx, key in enumerate(attr_keys):
                attr_last = idx == len(attr_keys) - 1
                attr_conn = "└── " if attr_last else "├── "
                val = obj.attrs[key]
                print(f"{attr_prefix}{attr_conn}@{key}: {val}")

        # 그룹인 경우 하위 항목 순회
        if isinstance(obj, h5py.Group):
            child_prefix = prefix + ("    " if is_last else "│   ")
            items = list(obj.items())
            for idx, (child_name, child_obj) in enumerate(items):
                _visit_node(
                    child_name,
                    child_obj,
                    child_prefix,
                    is_last=(idx == len(items) - 1),
                )

    # 파일 경로가 들어온 경우와 열린 객체가 들어온 경우 구분 처리
    if isinstance(file_path_or_obj, str):
        with h5py.File(file_path_or_obj, "r") as f:
            print(file_path_or_obj)
            items = list(f.items())
            for idx, (name, obj) in enumerate(items):
                _visit_node(name, obj, "", is_last=(idx == len(items) - 1))
    else:
        print(file_path_or_obj.name or "Group")
        items = list(file_path_or_obj.items())
        for idx, (name, obj) in enumerate(items):
            _visit_node(name, obj, "", is_last=(idx == len(items) - 1))

In [3]:
file = r"Z:\exp\hkim\261004\images\eiger_align.h5"
print_hdf5_tree(file)

Z:\exp\hkim\261004\images\eiger_align.h5
└── entry/
    ├── data/
    │   └── data [Dataset: shape=(1062, 1028), dtype=int32]
    └── instrument/
        ├── NDAttributes/
        │   ├── NDArrayEpicsTSSec [Dataset: shape=(1,), dtype=uint32]
        │   ├── NDArrayEpicsTSnSec [Dataset: shape=(1,), dtype=uint32]
        │   ├── NDArrayTimeStamp [Dataset: shape=(1,), dtype=float64]
        │   └── NDArrayUniqueId [Dataset: shape=(1,), dtype=int32]
        ├── detector/
        │   ├── NDAttributes/
        │   └── data [Dataset: shape=(1062, 1028), dtype=int32]
        └── performance/
            └── timestamp [Dataset: shape=(1, 5), dtype=float64]
